# eQTL-Style Multi-Trait LOCO MLM Acceleration

Practical pattern: many traits (e.g. gene expression) against the **same**
genotype matrix and a **shared** LOCO kinship.

1. Baseline: `PANICLE_MLM_LOCO` once per trait  
2. Accelerated: `PANICLE_MLM_LOCO_MULTI` (chromosome-major, reuses `U'G`)  
3. Write a compact QTL hit table  

This notebook is fully synthetic and does not require external data files.



## Imports



In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd

import panicle
from panicle.association.mlm_loco import PANICLE_MLM_LOCO, PANICLE_MLM_LOCO_MULTI
from panicle.matrix.kinship_loco import PANICLE_K_VanRaden_LOCO
from panicle.utils.data_types import GenotypeMatrix

print(f"PANICLE {panicle.__version__}")

HERE = Path.cwd().resolve()
if HERE.name == "examples" or (HERE / "eqtl_multitrait_acceleration_tutorial.ipynb").exists():
    OUT_DIR = HERE / "eqtl_tutorial_out"
elif (HERE / "examples").is_dir():
    OUT_DIR = HERE / "examples" / "eqtl_tutorial_out"
else:
    OUT_DIR = HERE / "eqtl_tutorial_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUT_DIR}")



## Build synthetic multi-trait data

- Shared individuals/genotypes for all traits  
- Multiple chromosomes in the map  
- One planted signal marker per trait  



In [ ]:
rng = np.random.default_rng(2026)

# Sized for a few-minute tutorial run on a laptop
n_individuals = 120
n_markers = 1200
n_traits = 4

geno = rng.integers(0, 3, size=(n_individuals, n_markers), dtype=np.int8)
geno_matrix = GenotypeMatrix(geno, is_imputed=True)

chrom_labels = np.array([str(i) for i in range(1, 5)])
chrom = np.repeat(chrom_labels, n_markers // chrom_labels.size)
if chrom.size < n_markers:
    chrom = np.concatenate([chrom, np.repeat(chrom_labels[-1], n_markers - chrom.size)])

map_df = pd.DataFrame({
    "MARKER": [f"M{i:05d}" for i in range(n_markers)],
    "CHROM": chrom,
    "POS": np.arange(1, n_markers + 1) * 100,
})

trait_names = [f"Gene_{i+1}" for i in range(n_traits)]
signal_markers = [40, 350, 700, 1050]

phe_matrix = np.zeros((n_individuals, n_traits), dtype=np.float64)
for t in range(n_traits):
    effect = 0.55 * geno[:, signal_markers[t]].astype(np.float64)
    noise = rng.normal(scale=1.0, size=n_individuals)
    phe_matrix[:, t] = effect + noise

covariates = rng.normal(size=(n_individuals, 2))

print(f"Individuals: {n_individuals}")
print(f"Markers: {n_markers}")
print(f"Traits: {n_traits}")
print(f"Planted signals at markers: {signal_markers}")



## Precompute shared LOCO kinship



In [ ]:
t0 = time.perf_counter()
loco = PANICLE_K_VanRaden_LOCO(
    geno_matrix,
    map_df,
    maxLine=512,
    cpu=1,
    verbose=False,
)
print(f"LOCO kinship: {time.perf_counter() - t0:.2f}s")
print(f"Chromosomes: {sorted(loco.chromosomes) if hasattr(loco, 'chromosomes') else 'n/a'}")



## Baseline: one trait at a time



In [ ]:
single_trait_results = {}
t0 = time.perf_counter()

for idx, trait_name in enumerate(trait_names):
    phe_single = np.column_stack([np.arange(n_individuals), phe_matrix[:, idx]])
    single_trait_results[trait_name] = PANICLE_MLM_LOCO(
        phe=phe_single,
        geno=geno_matrix,
        map_data=map_df,
        loco_kinship=loco,
        CV=covariates,
        maxLine=256,
        cpu=1,
        lrt_refinement=False,
        verbose=False,
    )

single_time = time.perf_counter() - t0
print(f"Per-trait loop runtime: {single_time:.2f}s")



## Accelerated: multi-trait chromosome-major path

`PANICLE_MLM_LOCO_MULTI` processes one chromosome at a time, computes the
expensive genotype transform once, and applies it to all traits.



In [ ]:
t0 = time.perf_counter()
multi_results = PANICLE_MLM_LOCO_MULTI(
    phe=phe_matrix,
    geno=geno_matrix,
    map_data=map_df,
    trait_names=trait_names,
    loco_kinship=loco,
    CV=covariates,
    maxLine=256,
    cpu=1,
    lrt_refinement=False,
    verbose=False,
)
multi_time = time.perf_counter() - t0
print(f"Grouped multi-trait runtime: {multi_time:.2f}s")
print(f"Speedup (single/multi): {single_time / max(multi_time, 1e-9):.2f}x")



## Numerical agreement



In [ ]:
rows = []
for trait_name in trait_names:
    a = single_trait_results[trait_name]
    b = multi_results[trait_name]
    rows.append({
        "trait": trait_name,
        "max_abs_effect_diff": float(np.nanmax(np.abs(a.effects - b.effects))),
        "max_abs_se_diff": float(np.nanmax(np.abs(a.se - b.se))),
        "max_abs_p_diff": float(np.nanmax(np.abs(a.pvalues - b.pvalues))),
    })

cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))

# Paths should agree closely (floating layout / blocking can introduce tiny diffs).
assert (cmp_df["max_abs_effect_diff"] < 1e-3).all(), cmp_df
assert (cmp_df["max_abs_p_diff"] < 1e-2).all(), cmp_df
print("Single-trait and multi-trait LOCO MLM agree within tutorial tolerances.")



## QTL hit table



In [ ]:
top_k = 5
all_hits = []
marker_values = map_df["MARKER"].to_numpy()
chrom_values = map_df["CHROM"].to_numpy()
pos_values = map_df["POS"].to_numpy()

for t_idx, trait_name in enumerate(trait_names):
    res = multi_results[trait_name]
    k = min(top_k, res.pvalues.size)
    top_idx = np.argpartition(res.pvalues, k - 1)[:k]
    top_idx = top_idx[np.argsort(res.pvalues[top_idx])]
    hit_df = pd.DataFrame({
        "Trait": trait_name,
        "MARKER": marker_values[top_idx],
        "CHROM": chrom_values[top_idx],
        "POS": pos_values[top_idx],
        "P_VALUE": res.pvalues[top_idx],
        "EFFECT": res.effects[top_idx],
        "SE": res.se[top_idx],
        "PLANTED_SIGNAL": marker_values[signal_markers[t_idx]],
    })
    all_hits.append(hit_df)

qtl_hits = pd.concat(all_hits, ignore_index=True)
output_path = OUT_DIR / "eqtl_multitrait_top_hits.csv"
qtl_hits.to_csv(output_path, index=False)
print(f"Wrote: {output_path}")
print(qtl_hits.head(12).to_string(index=False))

# Planted signals should rank among the strongest hits for each trait
for t_idx, trait_name in enumerate(trait_names):
    planted = map_df.loc[signal_markers[t_idx], "MARKER"]
    trait_hits = qtl_hits[qtl_hits["Trait"] == trait_name]
    assert planted in set(trait_hits["MARKER"]), (
        f"Planted signal {planted} not in top hits for {trait_name}"
    )
print("Planted signals recovered in top hits for every trait.")



## Notes

- Speedups grow with the number of traits that share samples and LOCO kinship.
- `lrt_refinement=False` isolates the Wald-path multi-trait acceleration; enable LRT for production top-hit calibration when needed.
- High-level `GWASPipeline` with multiple traits can also group LOCO work when sample masks match; this notebook shows the explicit multi-trait API.

